In [1]:
# ============================================================================
# APOLLO NLP MAIN COMPARISON – BERT-base on AG News (TIME-OPTIMIZED, 3 SEEDS)
#
# HOW THIS SCRIPT ADDRESSES REVIEWERS' COMMENTS:
# ----------------------------------------------------------------------------
# [EBM #1] Broaden practical scope -> Solved: Tests on large-scale text classification.
# [R2 #2] Modern benchmarks/Transformers -> Solved: Uses HuggingFace BERT-base.
# [R2 #3] Multiple seeds, means/std, stats -> Solved: Runs on 3 seeds, tracks CI, uses Paired T-Test + FDR.
# [R2 #4] Equal HP search budget -> Solved: Quick HP search applies identical grid to ALL optimizers.
# [R2 #5] Stability & complexity -> Solved: Generates Variance/CV% table and tracks `avg_epoch_time`.
# [R1 #6 & #7] Dataset ratio & source -> Solved: Code documents 120k/7.6k split & Kaggle dataset origin.
# [R1 #11] High resolution images -> Solved: All matplotlib figures saved at 300 DPI.
# [R1 #12] Proper labels -> Solved: LaTeX generators explicitly include dataset & model names in captions.
#
# QWEN FIXES: Fixed Fatal Statistical Paradox, Cleaned Metrics, Kaggle Checkpointing,
# Optimized DataLoaders, and Fixed standard Lion update formula.
# ============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import os, time, warnings, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.optimizer import Optimizer
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from transformers import BertTokenizerFast, BertForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import (f1_score, roc_auc_score, matthews_corrcoef, confusion_matrix)
from scipy.stats import ttest_rel, t as t_dist # [QWEN FIX & R2 #3 SOLUTION] Statistically powerful test for small N
from statsmodels.stats.multitest import multipletests

# ============================================================================
# 🛡️ ENVIRONMENT & REPRODUCIBILITY
# [REVIEWER R2 #3] "Evaluation should be conducted with multiple random seeds"
# SOLUTION: We explicitly define 3 distinct seeds for the main evaluation.
# ============================================================================
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
warnings.filterwarnings("ignore")
try:
    torch.set_float32_matmul_precision('high')
except:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Fixed seeds
HP_SEARCH_SEED = 42
MAIN_SEEDS = [42, 123, 587]   # [R2 #3 SOLUTION] 3 seeds for stronger statistical power

# Dataset & model paths (Kaggle offline)
AG_NEWS_PATH = '/kaggle/input/datasets/amananandrai/ag-news-classification-dataset'
BERT_PATH = '/kaggle/input/datasets/virajjayant/bertbaseuncased/bert-base-uncased'

# Output directory
RESULTS_DIR = "/kaggle/working/results_apollo_nlp_main"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "latex_tables"), exist_ok=True)

# [QWEN FIX] Dual Checkpoint Files to bypass Kaggle 12-hour limits
HP_CHECKPOINT_FILE = os.path.join(RESULTS_DIR, "hp_checkpoint.json")
MAIN_CHECKPOINT_FILE = os.path.join(RESULTS_DIR, "main_checkpoint.json")

# ============================================================================
# ⚙️ CONFIGURATION
# [REVIEWER R2 #4] "Ensure that all baselines receive an equal amount of search budget"
# SOLUTION: HP parameters are strictly defined here and applied uniformly.
# ============================================================================
NLP_EPOCHS = 2                 # full training epochs for main evaluation
BATCH_SIZE = 64
MAX_LENGTH = 128
GRADIENT_CLIP_VALUE = 1.0
WARMUP_RATIO = 0.1

# Quick HP search parameters (Equal Budget for ALL)
HP_EPOCHS = 1                  # only 1 epoch for search
HP_SUBSET_RATIO = 0.2          # use 20% of training data for speed
HP_LR_GRID = [2e-5, 5e-5]      # small grid
HP_WD_GRID = [0.01, 0.05]

# ============================================================================
# 🔄 REPRODUCIBILITY
# ============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================================
# 🧠 OPTIMIZERS
# [EDITOR EBM #2] "Novelty should be clarified... cosine similarity + EMA"
# SOLUTION: The Apollo class mathematically implements this exact mechanism.
# ============================================================================
class Apollo(Optimizer):
    def __init__(self, params, lr=2e-5, betas=(0.9, 0.999, 0.9), eps=1e-8,
                 weight_decay=0.01, total_steps=None):
        if not 0.0 <= lr: raise ValueError(f"Invalid lr: {lr}")
        self.total_steps = total_steps
        self.current_step = 0
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        self.current_step += 1
        for group in self.param_groups:
            beta1, beta2, beta3 = group['betas']
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                    state['update_agreement'] = torch.zeros((), device=p.device)
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])
                
                # [QWEN FIX] Standard Lion formulation inside Apollo
                lion_c = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                u_lion = torch.sign(lion_c)

                # AdamW formulation
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                m_hat = exp_avg / bc1
                v_hat = exp_avg_sq / bc2
                denom = v_hat.sqrt().add_(group['eps'])
                u_adam = m_hat / denom

                # Norm equalization for geometric validity
                adam_norm = u_adam.norm(p=2)
                lion_norm = u_lion.norm(p=2)
                if lion_norm > 0: u_lion = u_lion * (adam_norm / lion_norm)
                
                # Dynamic blending based on Cosine Similarity & EMA (beta3)
                cs_raw = F.cosine_similarity(u_adam.flatten(), u_lion.flatten(), dim=0, eps=1e-10)
                state['update_agreement'].mul_(beta3).add_(cs_raw, alpha=1 - beta3)
                bc3 = 1 - beta3 ** state['step']
                agreement_hat = state['update_agreement'] / bc3
                gamma = (agreement_hat + 1.0) / 2.0
                p.add_((1 - gamma) * u_adam + gamma * u_lion, alpha=-group['lr'])
        return loss

class Lion(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.01):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if group["weight_decay"] != 0:
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                if "exp_avg" not in state: state["exp_avg"] = torch.zeros_like(p)
                exp_avg = state["exp_avg"]
                beta1, beta2 = group["betas"]
                
                # Standard Lion update matching original paper
                update = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                p.add_(torch.sign(update), alpha=-group["lr"])
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)
        return loss

class LAMB(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-6, weight_decay=0.01):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                beta1, beta2 = group['betas']
                state['step'] += 1
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                m_hat = exp_avg / (1 - beta1 ** state['step'])
                v_hat = exp_avg_sq / (1 - beta2 ** state['step'])
                update = m_hat / v_hat.sqrt().add_(group['eps'])
                if group['weight_decay'] != 0: update.add_(p, alpha=group['weight_decay'])
                w_norm = p.norm(); g_norm = update.norm()
                trust_ratio = 1.0 if w_norm == 0 or g_norm == 0 else w_norm / g_norm
                p.add_(update, alpha=-group['lr'] * trust_ratio)
        return loss

class Sophia(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.965, 0.99), rho=0.04, weight_decay=0.1):
        defaults = dict(lr=lr, betas=betas, rho=rho, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['hessian'] = torch.zeros_like(p)
                exp_avg, hessian = state['exp_avg'], state['hessian']
                beta1, beta2 = group['betas']
                state['step'] += 1
                if group['weight_decay'] != 0:
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                hessian.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                update = torch.clip(exp_avg / (group['rho'] * hessian + 1e-15), -1, 1)
                p.add_(update, alpha=-group['lr'])
        return loss

# ============================================================================
# 📂 DATA LOADER (Optimized for Kaggle GPU)
# [REVIEWER R1 #6 & #7] Describe dataset, source, and ratio.
# SOLUTION: Dataset is AG News (120,000 Train / 7,600 Test). Loaded from Kaggle CSVs.
# ============================================================================
def load_agnews_data(tokenizer, max_length=128, subset_ratio=1.0, seed=42):
    train_df = pd.read_csv(os.path.join(AG_NEWS_PATH, 'train.csv'))
    test_df = pd.read_csv(os.path.join(AG_NEWS_PATH, 'test.csv'))
    
    for df in [train_df, test_df]:
        if 'Class Index' in df.columns: df.rename(columns={'Class Index': 'label'}, inplace=True)
        if 'Description' in df.columns and 'text' not in df.columns: df.rename(columns={'Description': 'text'}, inplace=True)
        elif 'description' in df.columns and 'text' not in df.columns: df.rename(columns={'description': 'text'}, inplace=True)
        if 'text' not in df.columns:
            label_col = 'label' if 'label' in df.columns else df.columns[0]
            df['text'] = df[[c for c in df.columns if c != label_col]].astype(str).agg(' '.join, axis=1)
            df.rename(columns={label_col: 'label'}, inplace=True)
        if df['label'].min() == 1: df['label'] -= 1

    class AGNewsDataset(Dataset):
        def __init__(self, dataframe, tokenizer, max_len):
            self.encodings = tokenizer(dataframe['text'].tolist(), truncation=True,
                                       padding='max_length', max_length=max_len, return_tensors='pt')
            self.labels = torch.tensor(dataframe['label'].values)
        def __len__(self): return len(self.labels)
        def __getitem__(self, idx):
            return {'input_ids': self.encodings['input_ids'][idx],
                    'attention_mask': self.encodings['attention_mask'][idx],
                    'labels': self.labels[idx]}

    full_train_ds = AGNewsDataset(train_df, tokenizer, max_length)
    val_ds = AGNewsDataset(test_df, tokenizer, max_length)

    if subset_ratio < 1.0:
        n_total = len(full_train_ds)
        n_subset = int(n_total * subset_ratio)
        indices = np.random.default_rng(seed).choice(n_total, n_subset, replace=False)
        train_ds = Subset(full_train_ds, indices)
    else:
        train_ds = full_train_ds

    # [QWEN FIX] Added pin_memory and 4 workers for max data ingestion speed
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    return train_loader, val_loader

# ============================================================================
# 🔄 TRAINING & EVALUATION
# [REVIEWER R2 #5] "Detailed discussion of convergence, stability, complexity"
# SOLUTION: We track train_loss (convergence), variance (stability), and avg_epoch_time (complexity)
# ============================================================================
def train_and_eval(model, train_loader, val_loader, optimizer, scheduler, num_epochs, use_amp=True):
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler() if use_amp else None

    best_val_loss = float('inf')
    best_state = None
    patience = 5
    no_improve = 0

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'epoch_time': []}

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        tr_loss, tr_correct, total = 0.0, 0, 0
        for batch in train_loader:
            # [QWEN FIX] non_blocking=True for faster tensor transfers
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            
            optimizer.zero_grad()
            if use_amp:
                with torch.cuda.amp.autocast():
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = criterion(outputs.logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                optimizer.step()
            
            scheduler.step()
            preds = outputs.logits.argmax(dim=1)
            tr_loss += loss.item() * labels.size(0)
            tr_correct += preds.eq(labels).sum().item()
            total += labels.size(0)

        epoch_time = time.time() - t0
        history['epoch_time'].append(epoch_time)
        history['train_loss'].append(tr_loss / total)
        history['train_acc'].append(tr_correct / total)

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
                val_loss += loss.item() * labels.size(0)
                val_correct += outputs.logits.argmax(1).eq(labels).sum().item()
                val_total += labels.size(0)
                
        history['val_loss'].append(val_loss / val_total)
        history['val_acc'].append(val_correct / val_total)

        if history['val_loss'][-1] < best_val_loss - 1e-4:
            best_val_loss = history['val_loss'][-1]
            no_improve = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
        if no_improve >= patience and epoch >= patience:
            if best_state: model.load_state_dict(best_state)
            break

    # Final evaluation
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_probs = np.vstack(all_probs)

    # [QWEN FIX] Removed unnecessary metrics (Prec, Spec, ALC) to avoid table clutter
    acc = float((all_preds == all_targets).mean())
    f1 = float(f1_score(all_targets, all_preds, average='macro', zero_division=0))
    try: auc_val = float(roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro'))
    except: auc_val = 0.0
    mcc = float(matthews_corrcoef(all_targets, all_preds))
    avg_epoch_time = float(np.mean(history['epoch_time']))

    return {'test_acc': acc, 'test_f1': f1, 'test_auc': auc_val,
            'test_mcc': mcc, 'avg_epoch_time': avg_epoch_time,
            'history': history}

# ============================================================================
# 💾 CHECKPOINTING UTILS
# ============================================================================
def load_json(filepath):
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r') as f: return json.load(f)
        except json.JSONDecodeError: pass
    return {}

def save_json(data, filepath):
    with open(filepath, 'w') as f: json.dump(data, f, indent=4)

# ============================================================================
# 📊 STATISTICS
# [REVIEWER R2 #3] "reporting means and standard deviations, statistical significance"
# SOLUTION: compute_statistics generates Mean/SD/CV. statistical_tests uses Paired T-Test.
# ============================================================================
def adaptive_ci(data, cl=0.95):
    n = len(data)
    if n < 5:
        mean, std = np.mean(data), np.std(data, ddof=1) if n > 1 else 0.0
        t_val = t_dist.ppf((1+cl)/2, max(n-1, 1))
        margin = t_val * std / np.sqrt(n) if n > 0 else 0
        return float(mean-margin), float(mean+margin)
    rng = np.random.default_rng(42)
    boot = [np.mean(rng.choice(data, n, replace=True)) for _ in range(1000)]
    alpha = (1-cl)/2
    return float(np.percentile(boot, 100*alpha)), float(np.percentile(boot, 100*(1-alpha)))

def compute_statistics(multi_results):
    metrics = ['test_acc', 'test_f1', 'test_auc', 'test_mcc', 'avg_epoch_time']
    agg, var, cvs, ci = {}, {}, {}, {}
    for opt, runs in multi_results.items():
        if not runs: continue
        agg[opt], var[opt], cvs[opt], ci[opt] = {}, {}, {}, {}
        for m in metrics:
            vals = [r[m] for r in runs]
            mean_v = float(np.mean(vals)); std_v = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            agg[opt][f'{m}_mean'] = mean_v; agg[opt][f'{m}_std'] = std_v
            var[opt][f'{m}_var'] = float(np.var(vals, ddof=1)) if len(vals) > 1 else 0.0
            cvs[opt][f'{m}_cv'] = float((std_v/mean_v*100) if mean_v>0 else 0)
            ci[opt][m] = adaptive_ci(vals)
    return agg, var, cvs, ci

def statistical_tests(results, baseline='Apollo'):
    if baseline not in results or len(results[baseline]) < 2: return {}
    base_vals = np.array([r['test_acc'] for r in results[baseline]])
    sig = {}; pvals = []; basenames = []
    
    for opt, runs in results.items():
        if opt == baseline or not runs or len(runs) < 2: continue
        vals = np.array([r['test_acc'] for r in runs])
        if len(base_vals)!=len(vals): continue
        
        basenames.append(opt)
        
        # [QWEN FIX] Mathematical stability for standard deviation of differences to avoid d_z paradox
        diffs = base_vals - vals
        mean_diff = float(np.mean(diffs))
        std_diff = float(np.std(diffs, ddof=1))
        
        if std_diff < 1e-8:
            d_z = 0.0
            p_raw = 1.0 if abs(mean_diff) < 1e-8 else 0.0
        else:
            d_z = float(mean_diff / std_diff)
            try: 
                # [R2 #3 SOLUTION] Paired T-test preferred for small sample sizes over Wilcoxon
                _, p_raw = ttest_rel(base_vals, vals)
            except: 
                p_raw = 1.0
                
        pvals.append(float(p_raw))
        sig[opt] = {'diff': mean_diff, 'p_raw': float(p_raw), 'es_z': d_z}
        
    if pvals:
        _, p_corr, _, _ = multipletests(pvals, method='fdr_bh')
        for i,opt in enumerate(basenames):
            sig[opt]['p_corr'] = float(p_corr[i])
            sig[opt]['sig'] = '***' if p_corr[i]<0.001 else '**' if p_corr[i]<0.01 else '*' if p_corr[i]<0.05 else 'ns'
    return sig

# ============================================================================
# 📄 LATEX TABLES & PLOTS
# [REVIEWER R1 #11] "High resolution image suggested"
# SOLUTION: plt.savefig uses dpi=300 for all outputs.
# [REVIEWER R1 #12] "Ensure Figures, Tables... are properly labeled"
# SOLUTION: All LaTeX tables and plot titles strictly include Dataset & Model names.
# ============================================================================
def performance_table(agg, ci, caption, label, filename):
    mc = [{'d':'Acc','k':'test_acc','f':'.4f'}, {'d':'F1','k':'test_f1','f':'.4f'},
          {'d':'AUC','k':'test_auc','f':'.4f'}, {'d':'MCC','k':'test_mcc','f':'.4f'},
          {'d':'Time(s)','k':'avg_epoch_time','f':'.2f'}]
    lines = [r"\begin{table}[htbp]", r"\centering", r"\caption{"+caption+"}", r"\label{"+label+"}",
             r"\resizebox{\textwidth}{!}{%", r"\begin{tabular}{l"+"c"*len(mc)+"}", r"\toprule",
             r"{\bf Optimizer} & "+" & ".join([f"{{\\bf {m['d']}}}" for m in mc])+r" \\", r"\midrule"]
    for opt in agg.keys():
        row = [opt]
        for m in mc:
            k,f = m['k'], m['f']
            mean = agg[opt][f'{k}_mean']; std = agg[opt][f'{k}_std']
            low, high = ci[opt][k]
            row.append(f"{mean:{f}} $\\pm$ {std:{f}} \\; [{low:{f}}, {high:{f}}]")
        lines.append(" & ".join(row)+r" \\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

def stats_table(sig, caption, label, filename):
    lines = [r"\begin{table}[htbp]", r"\centering", r"\caption{"+caption+"}", r"\label{"+label+"}",
             r"\begin{tabular}{lccccc}", r"\toprule",
             r"\textbf{Baseline} & \textbf{$\Delta$ Acc} & \textbf{$p_{\text{ttest}}$} & \textbf{$p_{\text{FDR}}$} & \textbf{$d_z$} & \textbf{Sig.} \\", r"\midrule"]
    for opt,r in sig.items():
        diff = r['diff']
        diff_str = f"${diff:+.4f}$" if abs(diff)>=1e-4 else f"${diff:+.1e}$"
        lines.append(f"{opt} & {diff_str} & {r['p_raw']:.3f} & {r['p_corr']:.3f} & {r['es_z']:.3f} & {r['sig']} \\\\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

def variance_table(var, cvs, caption, label, filename):
    metrics_show = [('test_acc','Accuracy'),('test_f1','F1'),('test_auc','AUC')]
    lines = [r"\begin{table}[htbp]", r"\centering", r"\caption{"+caption+"}", r"\label{"+label+"}",
             r"\begin{tabular}{l"+"cc"*len(metrics_show)+"}", r"\toprule",
             r"\multirow{2}{*}{\bf Optimizer} "+"".join([f"& \\multicolumn{{2}}{{c}}{{\\bf {disp}}}" for _,disp in metrics_show])+r" \\",
             r"\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7}",
             r" & "+" & ".join([r"$\sigma^2$ & CV\%"]*len(metrics_show))+r" \\", r"\midrule"]
    for opt in var.keys():
        row = [opt]
        for key,_ in metrics_show:
            v = var[opt][f'{key}_var']
            if v<1e-6: row.append(r"$<10^{-6}$")
            else: row.append(f"{v:.2e}")
            row.append(f"{cvs[opt][f'{key}_cv']:.2f}\\%")
        lines.append(" & ".join(row)+r" \\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

def plot_learning_curves(results_dict, title, save_name):
    plt.figure(figsize=(10,6))
    colors = plt.cm.tab10.colors
    for i,(label,runs) in enumerate(results_dict.items()):
        if not runs: continue
        accs = [r['history']['val_acc'] for r in runs]
        max_len = max(len(a) for a in accs)
        padded = [np.pad(a,(0,max_len-len(a)), constant_values=np.nan) for a in accs]
        mean_acc = np.nanmean(padded, axis=0); std_acc = np.nanstd(padded, axis=0)
        epochs = range(1, max_len+1)
        plt.plot(epochs, mean_acc, label=label, color=colors[i%len(colors)])
        plt.fill_between(epochs, mean_acc-std_acc, mean_acc+std_acc, alpha=0.2, color=colors[i%len(colors)])
    plt.xlabel('Epoch'); plt.ylabel('Validation Accuracy'); plt.title(title)
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    # [R1 #11 SOLUTION] Saved as high resolution (300 dpi) PNG
    plt.savefig(os.path.join(RESULTS_DIR, "plots", save_name), dpi=300, bbox_inches='tight')
    plt.close()

def plot_accuracy_barchart(agg, ci, title, save_name):
    optimizers = list(agg.keys())
    means = [agg[opt]['test_acc_mean'] for opt in optimizers]
    cis = [ci[opt]['test_acc'] for opt in optimizers]
    errors = [(m-c[0], c[1]-m) for m,c in zip(means,cis)]
    plt.figure(figsize=(10,6))
    x_pos = np.arange(len(optimizers))
    plt.bar(x_pos, means, yerr=np.array(errors).T, capsize=5,
            color=plt.cm.tab10.colors[:len(optimizers)], edgecolor='black', alpha=0.8)
    plt.xticks(x_pos, optimizers, rotation=15)
    plt.ylabel('Test Accuracy'); plt.title(title); plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    # [R1 #11 SOLUTION] Saved as high resolution (300 dpi) PNG
    plt.savefig(os.path.join(RESULTS_DIR, "plots", save_name), dpi=300, bbox_inches='tight')
    plt.close()

# ============================================================================
# 🔍 HYPERPARAMETER SEARCH (QUICK) & MAIN EVALUATION
# [REVIEWER R2 #4] "Clear hyperparameter search procedure, equal budget"
# SOLUTION: The quick_hyperparameter_search applies the exact same 1-epoch, 20% data grid
# across ALL baseline optimizers before doing the main 3-seed evaluation.
# ============================================================================

def quick_hyperparameter_search(tokenizer, optimizer_name, opt_class):
    hp_data = load_json(HP_CHECKPOINT_FILE)
    if optimizer_name in hp_data:
        print(f"  [Checkpoint] Skipped HP Search for {optimizer_name}. Loaded: lr={hp_data[optimizer_name]['lr']:.0e}, wd={hp_data[optimizer_name]['wd']}")
        return hp_data[optimizer_name]['lr'], hp_data[optimizer_name]['wd']
        
    print(f"  Quick HP search for {optimizer_name}...")
    train_loader, val_loader = load_agnews_data(tokenizer, MAX_LENGTH,
                                                subset_ratio=HP_SUBSET_RATIO, seed=HP_SEARCH_SEED)
    total_steps = HP_EPOCHS * len(train_loader)
    best_acc = 0.0
    best_params = (HP_LR_GRID[0], HP_WD_GRID[0])

    for lr in HP_LR_GRID:
        for wd in HP_WD_GRID:
            set_seed(HP_SEARCH_SEED)
            model = BertForSequenceClassification.from_pretrained(BERT_PATH, num_labels=4, local_files_only=True).to(device)
            if optimizer_name == 'Apollo':
                optimizer = opt_class(model.parameters(), lr=lr, weight_decay=wd, total_steps=total_steps)
            else:
                optimizer = opt_class(model.parameters(), lr=lr, weight_decay=wd)
            scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO*total_steps), total_steps)
            res = train_and_eval(model, train_loader, val_loader, optimizer, scheduler, HP_EPOCHS)
            acc = res['test_acc']
            if acc > best_acc:
                best_acc = acc
                best_params = (lr, wd)
            print(f"    lr={lr:.0e}, wd={wd} -> acc={acc:.4f}")
            
    print(f"  Best for {optimizer_name}: lr={best_params[0]:.0e}, wd={best_params[1]} (acc={best_acc:.4f})")
    
    # Save to checkpoint
    hp_data[optimizer_name] = {'lr': best_params[0], 'wd': best_params[1], 'acc': best_acc}
    save_json(hp_data, HP_CHECKPOINT_FILE)
    
    return best_params

def main_evaluation(tokenizer, best_configs):
    train_loader, val_loader = load_agnews_data(tokenizer, MAX_LENGTH, subset_ratio=1.0)
    total_steps = NLP_EPOCHS * len(train_loader)
    
    results = load_json(MAIN_CHECKPOINT_FILE)
    for name in best_configs:
        if name not in results: results[name] = []

    for seed in MAIN_SEEDS:
        for opt_name, (opt_class, lr, wd) in best_configs.items():
            if len([r for r in results[opt_name] if r.get('seed') == seed]) > 0:
                print(f"  [Checkpoint] Skipping Main Eval for {opt_name} seed={seed} (Already completed)")
                continue
                
            set_seed(seed)
            model = BertForSequenceClassification.from_pretrained(BERT_PATH, num_labels=4, local_files_only=True).to(device)
            if opt_name == 'Apollo':
                optimizer = opt_class(model.parameters(), lr=lr, weight_decay=wd, total_steps=total_steps)
            else:
                optimizer = opt_class(model.parameters(), lr=lr, weight_decay=wd)
            scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO*total_steps), total_steps)
            
            res = train_and_eval(model, train_loader, val_loader, optimizer, scheduler, NLP_EPOCHS)
            res['seed'] = seed
            print(f"  {opt_name} seed={seed} acc={res['test_acc']:.4f}")
            
            results[opt_name].append(res)
            save_json(results, MAIN_CHECKPOINT_FILE)
            
    return results

# ============================================================================
# 🚀 MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("=" * 80)
    print("APOLLO NLP MAIN COMPARISON – Kaggle Optimized & Statistically Robust")
    print("=" * 80)

    # [R2 #2 SOLUTION] Using modern architecture (Transformer: BERT-base)
    tokenizer = BertTokenizerFast.from_pretrained(BERT_PATH, local_files_only=True)

    # --- Step 1: Quick HP search for all optimizers (equal budget) ---
    hp_configs = {}
    hp_configs['Apollo'] = (Apollo, *quick_hyperparameter_search(tokenizer, 'Apollo', Apollo))
    hp_configs['AdamW'] = (optim.AdamW, *quick_hyperparameter_search(tokenizer, 'AdamW', optim.AdamW))
    hp_configs['Lion'] = (Lion, *quick_hyperparameter_search(tokenizer, 'Lion', Lion))
    hp_configs['LAMB'] = (LAMB, *quick_hyperparameter_search(tokenizer, 'LAMB', LAMB))
    hp_configs['Sophia'] = (Sophia, *quick_hyperparameter_search(tokenizer, 'Sophia', Sophia))

    print("\nBest hyperparameters for Main Evaluation:")
    for name, (_, lr, wd) in hp_configs.items():
        print(f"  {name}: lr={lr:.0e}, wd={wd}")

    # --- Step 2: Main evaluation with 3 seeds (full data, 2 epochs) ---
    print("\nStarting Main Evaluation...")
    main_results = main_evaluation(tokenizer, hp_configs)

    # --- Step 3: Statistics, tables, plots ---
    agg, var, cvs, ci = compute_statistics(main_results)
    sig = statistical_tests(main_results, baseline='Apollo')

    # [R1 #12 SOLUTION] Explicit captions in all tables
    performance_table(agg, ci, "NLP Main Comparison (BERT-base on AG News, Mean $\\pm$ SD [95\\% CI])",
                      "tab:nlp_main_perf", "t1_nlp_main_perf.tex")
    
    stats_table(sig, "Statistical Significance (Apollo vs baselines, Paired T-Test) - BERT-base on AG News",
                "tab:nlp_main_stats", "t2_nlp_main_stats.tex")
                
    variance_table(var, cvs, "Stability Analysis (Variance \& CV) - BERT-base on AG News",
                   "tab:nlp_main_var", "t3_nlp_main_var.tex")
                   
    plot_learning_curves(main_results, "NLP Learning Curves (BERT-base on AG News)", "nlp_learning_curves.png")
    plot_accuracy_barchart(agg, ci, "NLP Final Accuracy Comparison (BERT-base on AG News)", "nlp_accuracy_barchart.png")

    print("\n✅ NLP MAIN COMPARISON COMPLETE. Checkpoints, Results and LaTeX tables saved.")

<>:691: SyntaxWarning: invalid escape sequence '\&'
<>:691: SyntaxWarning: invalid escape sequence '\&'
/tmp/ipykernel_23/3627824528.py:691: SyntaxWarning: invalid escape sequence '\&'
  variance_table(var, cvs, "Stability Analysis (Variance \& CV) - BERT-base on AG News",


Device: cuda
APOLLO NLP MAIN COMPARISON – Kaggle Optimized & Statistically Robust
  Quick HP search for Apollo...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/virajjayant/bertbaseuncased/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because

    lr=2e-05, wd=0.01 -> acc=0.9083


...

BertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/virajjayant/bertbaseuncased/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because

  LAMB seed=587 acc=0.9341


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/virajjayant/bertbaseuncased/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because

  Sophia seed=587 acc=0.9351

✅ NLP MAIN COMPARISON COMPLETE. Checkpoints, Results and LaTeX tables saved.
